#Demonstration: Building an NLP Pipeline for Multilingual Tweet Cleaning

##Scenario:

- Alex, an NLP engineer at a social media company, receives a messy dataset of multilingual tweets filled with emojis, emoticons, hashtags, mentions, and noise. Since the goal is to analyze only English tweets, manually cleaning and filtering them would be too time-consuming. To streamline this, she builds an automated NLP pipeline that detects language, filters for English, removes noise, lowercases text, eliminates stopwords, tokenizes content, and applies techniques like BPE, one-hot encoding, and TF-IDF—turning raw tweets into clean, structured data ready for analysis.

##Import Libraries

In [3]:
!pip install nltk langdetect spacy ftfy contractions emoji tokenizers --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [10]:
# Core Libraries
import pandas as pd
import re
import string
import unicodedata
import ftfy
import contractions

# Language & Tokenization
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from langdetect import detect
import spacy

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

# Emojis & Emoticons
import emoji

# Byte Pair Encoding (BPE)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Download resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Load SpaCy
nlp = spacy.load("en_core_web_sm")
print("✅ Libraries imported and models downloaded.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Libraries imported and models downloaded.


##Load and Show Dataset

In [5]:
df = pd.read_csv("multilingual_twitter_dataset.csv")
print("📄 Sample data:\n", df.head())

📄 Sample data:
   username location  gender  age  \
0   user_1    India  Female   43   
1   user_2   France    Male   26   
2   user_3  Germany   Other   30   
3   user_4  Germany  Female   38   
4   user_5    Japan  Female   39   

                                           tweet  
0  Can't believe this happened... lol :D #fail 😅  
1                                     信じられない… 😭💔  
2                 Das ist fantastisch 😍💯! #Liebe  
3  Can't believe this happened... lol :D #fail 😅  
4                  ¡Esto es perfecto! 😊💃 #fiesta  


##Detect Languages

In [6]:
df['language'] = df['tweet'].apply(lambda x: detect(x))
print("🗣️ Language distribution:\n", df['language'].value_counts())

🗣️ Language distribution:
 language
de    14
pt    14
id    14
fr    12
ja    12
en     8
ru     8
hi     7
es     6
ur     4
ar     1
Name: count, dtype: int64


##Filter English Tweets

In [7]:
df_en = df[df['language'] == 'en'].reset_index(drop=True)
print("🔤 English-only Tweets:\n", df_en[['tweet']].head())

🔤 English-only Tweets:
                                            tweet
0  Can't believe this happened... lol :D #fail 😅
1  Can't believe this happened... lol :D #fail 😅
2  Can't believe this happened... lol :D #fail 😅
3        I love this! 😍 Soooo good!!! #awesome 😊
4  Can't believe this happened... lol :D #fail 😅


##Lowercasing

In [12]:
df_en['cleaned'] = df_en['tweet'].apply(lambda x: x.lower())
print("🔡 Lowercased sample:\n", df_en['cleaned'].head())

🔡 Lowercased sample:
 0    can't believe this happened... lol :d #fail 😅
1    can't believe this happened... lol :d #fail 😅
2    can't believe this happened... lol :d #fail 😅
3          i love this! 😍 soooo good!!! #awesome 😊
4    can't believe this happened... lol :d #fail 😅
Name: cleaned, dtype: object


##Stopword Removal

In [13]:
stop_words = set(stopwords.words('english'))
df_en['cleaned'] = df_en['cleaned'].apply(
    lambda x: ' '.join([word for word in word_tokenize(x) if word not in stop_words])
)
print("🧹 After stopword removal:\n", df_en['cleaned'].head())

🧹 After stopword removal:
 0    ca n't believe happened ... lol : # fail 😅
1    ca n't believe happened ... lol : # fail 😅
2    ca n't believe happened ... lol : # fail 😅
3         love ! 😍 soooo good ! ! ! # awesome 😊
4    ca n't believe happened ... lol : # fail 😅
Name: cleaned, dtype: object


##Tokenization

In [14]:
df_en['tokens'] = df_en['cleaned'].apply(word_tokenize)
print("✂️ Tokens:\n", df_en['tokens'].head())

✂️ Tokens:
 0    [ca, n't, believe, happened, ..., lol, :, #, f...
1    [ca, n't, believe, happened, ..., lol, :, #, f...
2    [ca, n't, believe, happened, ..., lol, :, #, f...
3    [love, !, 😍, soooo, good, !, !, !, #, awesome, 😊]
4    [ca, n't, believe, happened, ..., lol, :, #, f...
Name: tokens, dtype: object


##Byte-Pair Encoding (BPE)

In [15]:
# Prepare file for BPE training
with open("bpe_corpus.txt", "w", encoding="utf-8") as f:
    for text in df_en['cleaned']:
        f.write(text + "\n")

# Train and tokenize using BPE
tokenizer = Tokenizer(BPE())
trainer = BpeTrainer(special_tokens=["<unk>"])
tokenizer.pre_tokenizer = Whitespace()
tokenizer.train(files=["bpe_corpus.txt"], trainer=trainer)
df_en['bpe_tokens'] = df_en['cleaned'].apply(lambda x: tokenizer.encode(x).tokens)

print("🧠 BPE token sample:\n", df_en['bpe_tokens'].head())

🧠 BPE token sample:
 0    [ca, n, ', t, believe, happened, ..., lol, :, ...
1    [ca, n, ', t, believe, happened, ..., lol, :, ...
2    [ca, n, ', t, believe, happened, ..., lol, :, ...
3    [love, !, 😍, soooo, good, !, !, !, #, awesome, 😊]
4    [ca, n, ', t, believe, happened, ..., lol, :, ...
Name: bpe_tokens, dtype: object


##One-Hot Encoding

In [16]:
# Flatten all words into one list
all_words = list(set([word for tokens in df_en['tokens'] for word in tokens]))

# Create encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit([[word] for word in all_words])

# Encode first tweet for demo
encoded_sample = encoder.transform([[w] for w in df_en['tokens'][0]])
print("🎯 One-Hot for 1st Tweet (shape):", encoded_sample.shape)

🎯 One-Hot for 1st Tweet (shape): (10, 17)


##TF-IDF Vectorization

In [19]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_en['cleaned'])
print("📈 TF-IDF Shape:", tfidf_matrix.shape)

📈 TF-IDF Shape: (8, 9)


##Remove Noise (URLs, Mentions, Hashtags, Punctuation)

In [20]:
def remove_noise(text):
    text = re.sub(r"http\S+|www\S+", "", text)      # URLs
    text = re.sub(r"@\w+", "", text)                # Mentions
    text = re.sub(r"#\w+", "", text)                # Hashtags
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)  # Punctuation
    return text

df_en['cleaned'] = df_en['cleaned'].apply(remove_noise)
print("🚮 After noise removal:\n", df_en['cleaned'].head())

🚮 After noise removal:
 0    ca nt believe happened  lol   fail 😅
1    ca nt believe happened  lol   fail 😅
2    ca nt believe happened  lol   fail 😅
3        love  😍 soooo good     awesome 😊
4    ca nt believe happened  lol   fail 😅
Name: cleaned, dtype: object


##Remove Emojis

In [21]:
def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

df_en['cleaned'] = df_en['cleaned'].apply(remove_emojis)
print("😀 Emojis removed:\n", df_en['cleaned'].head())

😀 Emojis removed:
 0    ca nt believe happened  lol   fail 
1    ca nt believe happened  lol   fail 
2    ca nt believe happened  lol   fail 
3         love   soooo good     awesome 
4    ca nt believe happened  lol   fail 
Name: cleaned, dtype: object


##Remove Emoticons

In [22]:
emoticons = {":)", ":(", ":D", ":-)", ":-(", ":'(", ":-D", ";)", ":P", ":-P"}

def remove_emoticons(text):
    return ' '.join([word for word in text.split() if word not in emoticons])

df_en['cleaned'] = df_en['cleaned'].apply(remove_emoticons)
print("😐 Emoticons removed:\n", df_en['cleaned'].head())

😐 Emoticons removed:
 0    ca nt believe happened lol fail
1    ca nt believe happened lol fail
2    ca nt believe happened lol fail
3            love soooo good awesome
4    ca nt believe happened lol fail
Name: cleaned, dtype: object


##Handle Contractions

In [24]:
df_en['cleaned'] = df_en['cleaned'].apply(lambda x: contractions.fix(x))
print("🔧 Contractions handled:\n", df_en['cleaned'].head())

🔧 Contractions handled:
 0    ca nt believe happened lol fail
1    ca nt believe happened lol fail
2    ca nt believe happened lol fail
3            love soooo good awesome
4    ca nt believe happened lol fail
Name: cleaned, dtype: object


##Normalization (Accents, Spaces)

In [26]:
def normalize_text(text):
    text = unicodedata.normalize('NFKC', text)  # Unicode normalization
    text = ftfy.fix_text(text)  # Fix messy text
    return text

df_en['cleaned'] = df_en['cleaned'].apply(normalize_text)
print("🧽 Normalized text:\n", df_en['cleaned'].head())

🧽 Normalized text:
 0    ca nt believe happened lol fail
1    ca nt believe happened lol fail
2    ca nt believe happened lol fail
3            love soooo good awesome
4    ca nt believe happened lol fail
Name: cleaned, dtype: object


##Unicode Normalization

In [27]:
def unicode_normalize(text):
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')

df_en['cleaned'] = df_en['cleaned'].apply(unicode_normalize)
print("🔤 Unicode normalized:\n", df_en['cleaned'].head())

🔤 Unicode normalized:
 0    ca nt believe happened lol fail
1    ca nt believe happened lol fail
2    ca nt believe happened lol fail
3            love soooo good awesome
4    ca nt believe happened lol fail
Name: cleaned, dtype: object


##Display Final Result

In [28]:
print("✅ Final Processed Output Sample:\n", df_en[['tweet', 'cleaned', 'tokens', 'bpe_tokens']].head())

✅ Final Processed Output Sample:
                                            tweet  \
0  Can't believe this happened... lol :D #fail 😅   
1  Can't believe this happened... lol :D #fail 😅   
2  Can't believe this happened... lol :D #fail 😅   
3        I love this! 😍 Soooo good!!! #awesome 😊   
4  Can't believe this happened... lol :D #fail 😅   

                           cleaned  \
0  ca nt believe happened lol fail   
1  ca nt believe happened lol fail   
2  ca nt believe happened lol fail   
3          love soooo good awesome   
4  ca nt believe happened lol fail   

                                              tokens  \
0  [ca, n't, believe, happened, ..., lol, :, #, f...   
1  [ca, n't, believe, happened, ..., lol, :, #, f...   
2  [ca, n't, believe, happened, ..., lol, :, #, f...   
3  [love, !, 😍, soooo, good, !, !, !, #, awesome, 😊]   
4  [ca, n't, believe, happened, ..., lol, :, #, f...   

                                          bpe_tokens  
0  [ca, n, ', t, believe, happen

##Save Cleaned Dataset

In [29]:
df_en.to_csv("cleaned_multilingual_tweets.csv", index=False)
print("📁 Cleaned dataset saved as 'cleaned_multilingual_tweets.csv'")


📁 Cleaned dataset saved as 'cleaned_multilingual_tweets.csv'
